# Guia da Aula 03 — Métodos Baseados em Políticas

**Disciplina:** Modelos de Aprendizagem por Reforço  
**Aula:** 03 — Policy-Based Methods  
**Formato:** Guia de navegação e validação de ambiente

| | |
|---|---|
| **Aula** | Aula 03 — Métodos Baseados em Políticas |
| **Notebook** | 00 — Guia da Aula |
| **Seções** | — |
| **Tempo de leitura** | ~5 min |
| **Tempo de execução** | ~1 min |

**Pré-requisitos:** Aula 02 completa (Value-Based Methods, Q-Learning, DQN).

**Competências para o Desafio Final:** Orientação sobre os notebooks da aula e validação do ambiente de execução.

### Recapitulando

A Aula 02 percorreu seis métodos baseados em valor — de Programação Dinâmica ao DQN. A limitação central que ficou em aberto: todos assumem que o espaço de ações é *discreto e finito*, pois derivam a política via `argmax_a Q(s,a)`. Em robótica e controle, as ações são contínuas — essa operação se torna inviável. A Aula 03 responde diretamente a essa limitação ao parametrizar a política como o objeto de aprendizado.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import rl_utils
rl_utils.info_versoes()
rl_utils.configurar_matplotlib()
rl_utils.definir_seeds(42)

Biblioteca           Versão
--------------------------------
gymnasium            1.0.0


torch                2.11.0+cpu
numpy                2.4.2
matplotlib           3.10.8
pandas               3.0.1


scikit-learn         1.8.0


## Bloco 1 — Contexto e pergunta central

A Aula 02 percorreu uma família inteira de métodos — de Programação Dinâmica ao DQN — todos com o mesmo ponto cego: **o espaço de ações precisa ser discreto e enumerável**. A operação `max_a Q(s,a)` pressupõe que é possível comparar o valor de cada ação. Em robótica, controle de veículos e muitos problemas reais, as ações são contínuas — e essa operação deixa de ser viável.

A Aula 03 responde a esse desafio com uma mudança de objeto: em vez de aprender *quanto vale* cada ação, o agente aprende *diretamente como se comportar*. A política deixa de ser derivada de uma função de valor e passa a ser o objeto central de aprendizado, otimizada por gradiente.

> **"Como um agente aprende a ajustar diretamente seu comportamento para maximizar o retorno esperado?"**

O fio condutor da aula é a progressão:

> intuição de política → Policy Gradient → baselines → Actor-Critic → PPO → controle contínuo (DDPG e SAC)

## Bloco 2 — Mapa da aula e ordem de execução

Execute os notebooks **na ordem numérica** (00 → 06). Cada notebook é autocontido, mas a progressão conceitual faz mais sentido quando seguida em ordem.

| Notebook | Tema | Ambiente | Seções do plano |
|---|---|---|---|
| `aula03_00_guia.ipynb` | Guia de navegação + validação de ambiente | — | — |
| `aula03_01_intuicao_policy.ipynb` | Motivação para policy-based, política parametrizada | CartPole-v1 | 3.1 |
| `aula03_02_policy_gradient.ipynb` | REINFORCE, gradiente de política, alta variância | CartPole-v1 | 3.2 |
| `aula03_03_baseline_actor_critic.ipynb` | Baselines, função de vantagem, Actor-Critic (A2C) | CartPole-v1 | 3.3, 3.4 |
| `aula03_04_ppo.ipynb` | Trust regions, PPO com clipping, surrogate objective | CartPole-v1 | 3.5, 3.6 |
| `aula03_05_controle_continuo_ddpg_sac.ipynb` | DDPG, SAC, controle contínuo | Pendulum-v1 | 3.7, 3.8 |
| `aula03_06_comparativo_final.ipynb` | Value-based vs policy-based, tabela síntese, transição Aula 04 | CartPole-v1 | 3.9 |

> **Atenção — NB05:** `stable-baselines3` é **opcional**. O notebook implementa DDPG do zero no Pendulum-v1; a análise de SAC usa SB3 se disponível, caso contrário apresenta os resultados de forma conceitual.

## Como executar os notebooks

**Ambiente Python:** Python ≥ 3.9. Ative o ambiente virtual e instale as dependências:

    source .venv/bin/activate
    pip install -r requirements.txt
    jupyter lab

**Dependência opcional para o Notebook 05 (SAC via Stable-Baselines3):**

    pip install stable-baselines3

**Reprodutibilidade:** semente aleatória fixada em todos os notebooks (`SEED = 42`). Os resultados podem variar ligeiramente entre versões de bibliotecas.

| Notebook | Tempo estimado (CPU) |
|---|---|
| 00 — Guia | < 1 min |
| 01 — Intuição de política | 1–2 min |
| 02 — REINFORCE | 2–4 min |
| 03 — Baseline / Actor-Critic | 3–5 min |
| 04 — PPO | 3–6 min |
| 05 — DDPG / SAC | 5–15 min |
| 06 — Comparativo | 2–4 min |

In [2]:
# Instalação opcional das dependências principais
# %pip install numpy matplotlib gymnasium torch

# Para o Notebook 05 (SAC via Stable-Baselines3), instalar também:
# %pip install stable-baselines3

In [3]:
import sys
import importlib

print(f"Python: {sys.version}")
print()

bibliotecas = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "gymnasium": "gymnasium",
    "torch": "torch",
}

for nome, modulo in bibliotecas.items():
    try:
        m = importlib.import_module(modulo)
        versao = getattr(m, "__version__", "versão não disponível")
        print(f"  {nome}: {versao}")
    except ImportError:
        print(f"  {nome}: NÃO INSTALADO")

# stable-baselines3 — dependência opcional do Notebook 05
try:
    import stable_baselines3
    print(f"  stable-baselines3: {stable_baselines3.__version__}")
except ImportError:
    print("  stable-baselines3: não instalado (opcional — apenas para NB05 com SAC)")

print()
print("Ambiente validado. Execute os notebooks na ordem: 00 → 01 → 02 → 03 → 04 → 05 → 06")

Python: 3.13.13 (main, Apr  8 2026, 09:49:30) [GCC 13.3.0]

  numpy: 2.4.2
  matplotlib: 3.10.8
  gymnasium: 1.0.0
  torch: 2.11.0+cpu
  stable-baselines3: não instalado (opcional — apenas para NB05 com SAC)

Ambiente validado. Execute os notebooks na ordem: 00 → 01 → 02 → 03 → 04 → 05 → 06


## Bloco 3 — Leitura dos resultados da validação

A célula acima imprime as versões instaladas. Interpretação:

- `numpy`, `matplotlib`, `gymnasium`, `torch` — obrigatórios para os Notebooks 01 a 06.
- `stable-baselines3` — opcional. Se ausente, o Notebook 05 apresenta DDPG implementado do zero e descreve o SAC de forma conceitual, sem perda do conteúdo principal.

Caso qualquer biblioteca obrigatória apareça como `NÃO INSTALADO`, execute `pip install -r requirements.txt` com o ambiente virtual ativado antes de prosseguir.

## Bloco 4 — Interpretação pedagógica da progressão

Cada notebook responde a uma versão mais sofisticada da mesma pergunta central. A progressão foi projetada para que o aluno perceba por que cada limitação anterior exige uma nova abordagem:

- **Notebook 01** constrói a intuição: a política como distribuição de probabilidade sobre ações, parametrizada e otimizável. O agente não pergunta *qual estado vale mais*, mas *como se comportar*.
- **Notebook 02** implementa o REINFORCE: o gradiente reforça ações com bons retornos. A alta variância das atualizações aparece na curva de treinamento como o problema central a ser resolvido.
- **Notebook 03** responde à variância com baselines e a função de vantagem. Introduz a arquitetura Actor-Critic, onde o crítico estima o valor do estado para orientar o ator.
- **Notebook 04** mostra que atualizações de política muito grandes podem destruir o que foi aprendido. O PPO resolve isso com *clipping* do objetivo substituto, tornando o treinamento estável.
- **Notebook 05** entra no controle contínuo: ações que não são categorias, mas valores reais. O DDPG usa uma política determinística com ruído de exploração externo; o SAC incorpora a entropia como parte do objetivo.
- **Notebook 06** consolida a comparação entre as duas grandes famílias — value-based (Aula 02) e policy-based (Aula 03) — e aponta para a Aula 04.

---

**Sobre o critério de convergência do CartPole-v1:** os Notebooks 01 a 04 e 06 usam o CartPole-v1 como ambiente de referência. O limiar de convergência adotado é **retorno médio ≥ 475 nos últimos 100 episódios**. Esse número corresponde a manter o mastro em pé por pelo menos 475 dos 500 passos máximos do episódio — praticamente a duração completa, indicando controle efetivo. O Notebook 05 usa o Pendulum-v1, cujo espaço de ações é contínuo; o critério de qualidade é diferente (retorno ≥ −200, sendo −1600 o pior caso e −120 o ótimo).

## Bloco 5 — Limites deste guia e próximo passo

Este notebook não ensina nenhum algoritmo; sua função é orientar a navegação e confirmar que o ambiente está pronto.

O aprendizado começa no **Notebook 01 (`aula03_01_intuicao_policy.ipynb`)**, com a motivação para aprender políticas diretamente — o ponto de partida mais concreto para responder à pergunta central desta aula.

## Conexão com o problema de recomendação MovieLens

Os métodos policy-based desta aula resolvem uma limitação que ficou em aberto na Aula 01: o catálogo de filmes é tipicamente grande demais para enumerar todas as ações via `argmax Q(s,a)`. Políticas parametrizadas permitem decidir *diretamente* qual item recomendar a partir da representação do usuário.

| Algoritmo | Aplicação em recomendação |
|---|---|
| **REINFORCE** | Aprender uma política de seleção de item sem Q(s,a) tabular — adequado quando o catálogo inviabiliza a tabela |
| **A2C / PPO** | Treinar agente de recomendação que balanceia exploração e aproveitamento de forma estável ao longo do tempo |
| **DDPG / SAC** | Aprender políticas sobre embeddings contínuos de usuários e filmes — o espaço de ações é o espaço latente do catálogo |

**Conexão prática:** sistemas de recomendação industriais (Netflix, Spotify) usam variantes de policy gradient para otimizar métricas de longo prazo — retenção, diversidade, engajamento — não apenas a probabilidade de clique imediato. Os algoritmos desta aula são o ponto de partida conceitual para essas aplicações.

## Mapeamento para o Desafio Final

| Notebook | Competência construída | Quando usar no Desafio Final |
|---|---|---|
| NB01 — Intuição de política | Política parametrizada, softmax, diferença value-based vs policy-based | Qualquer tarefa com espaço de ações grande ou contínuo |
| NB02 — REINFORCE | Gradiente de política, log-probabilidade, alta variância | Baseline de referência — algoritmo mais simples de implementar |
| NB03 — Baseline e A2C | Função de vantagem, redução de variância, arquitetura Actor-Critic | Quando REINFORCE for instável; base para todos os algoritmos modernos |
| NB04 — PPO | Clipping do objetivo surrogate, múltiplas épocas, estabilidade | Ações discretas em qualquer problema de controle |
| NB05 — DDPG e SAC | Controle contínuo, replay buffer, política determinística/estocástica | Ações contínuas — torques, velocidades, ângulos; robótica |
| NB06 — Comparativo | Escolha do algoritmo adequado ao tipo de ação e restrição de dados | Diagnóstico: seleção de método antes de implementar no Desafio Final |

**Guia de decisão em 3 perguntas:**

1. **Ações discretas ou contínuas?** Discretas → PPO (NB04). Contínuas → SAC (NB05).
2. **Dados limitados ou ambiente caro de simular?** Sim → SAC/DDPG off-policy com replay buffer.
3. **Recompensa esparsa ou horizonte muito longo?** Considerar métodos hierárquicos (Aula 04, NB05) antes de tunar hiperparâmetros.

In [4]:
# Glossário dos termos técnicos deste notebook
rl_utils.exibir_glossario([
    'agent', 'environment', 'state', 'action', 'reward',
    'episode', 'trajectory', 'return', 'policy', 'policy gradient',
])

Termo (EN)        Tradução (PT)                Descrição
---------------------------------------------------------------------------------------------------------
action            ação                         Decisão tomada pelo agente em cada passo.
agent             agente                       Entidade que toma decisões no ambiente.
environment       ambiente                     Sistema com o qual o agente interage.
episode           episódio                     Sequência completa de interação do início ao estado terminal.
policy            política                     π(a|s) — distribuição de probabilidade sobre ações dado o estado.
policy gradient   gradiente de política        Família de métodos que otimiza diretamente os parâmetros da política.
return            retorno                      Soma (descontada) de recompensas futuras a partir de um estado.
reward            recompensa                   Sinal escalar de feedback do ambiente ao agente.
state             estado      

## Leituras e referências

- Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2ª ed.). MIT Press. Cap. 13 (Policy Gradient Methods). Disponível em: http://incompleteideas.net/book/the-book-2nd.html. Acesso em: abril 2026.
- Schulman, J., Wolski, F., Dhariwal, P., Radford, A., & Klimov, O. (2017). Proximal Policy Optimization Algorithms. *arXiv:1707.06347*. Disponível em: https://arxiv.org/abs/1707.06347. Acesso em: abril 2026.
- Haarnoja, T., Zhou, A., Abbeel, P., & Levine, S. (2018). Soft Actor-Critic: Off-Policy Maximum Entropy Deep Reinforcement Learning with a Stochastic Actor. *arXiv:1801.01290*. Disponível em: https://arxiv.org/abs/1801.01290. Acesso em: abril 2026.
- Farama Foundation. *Gymnasium documentation*. Disponível em: https://gymnasium.farama.org. Acesso em: abril 2026.